# 02 - ColPali Indexing & Retrieval (Google Colab / GPU)

ColPali (PaliGemma-3B base, multi-vector) is too heavy to run on an 8GB M1,
so its index is built **here on a Colab GPU**. The resulting index is then
downloaded and dropped into `data/index/colpali/` locally, where the rest of
the system (QA mode, comparison) consumes it like any other retriever.

**Runtime:** set Colab to a GPU runtime (`Runtime -> Change runtime type -> T4 GPU`).

Steps: install deps -> get the repo + data -> build the ColPali index ->
compare ColPali vs BiomedCLIP -> download the index.

In [ ]:
# 1. Install dependencies (ColPali is NOT in requirements.txt - it is Colab-only).
!pip install -q "colpali-engine>=0.3.0,<0.4.0" open_clip_torch faiss-cpu \
    transformers accelerate pyyaml python-dotenv pandas pillow tqdm sacrebleu rouge-score

In [ ]:
# 2. Get the project code. Either clone your GitHub repo or upload the folder.
#    Replace the URL with your repository.
# !git clone https://github.com/<your-username>/cxr-intelligence.git
# %cd cxr-intelligence

import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() / 'src'))
print('cwd:', pathlib.Path.cwd())

## 3 - Data

Upload `data/raw/` (the MIMIC-CXR CSV + images) and `data/qa/qa_dataset.json`,
or re-run `scripts/download_data.py` here with Kaggle credentials.

In [ ]:
from cxr.config import CONFIG
from cxr.data.loader import load_records

records = load_records(limit=CONFIG.qa_generation.max_reports)
print(f'Loaded {len(records)} reports.')

## 4 - Build the ColPali index (renders each report as a page image)

In [ ]:
from cxr.models.retrievers import build_retriever

colpali = build_retriever('colpali')
colpali.build_index(records)   # writes data/index/colpali/

## 5 - Compare ColPali vs BiomedCLIP on the QA eval set

In [ ]:
import random
import pandas as pd
from cxr.data.qa_builder import load_qa_dataset
from cxr.evaluation.retrieval_eval import evaluate_retriever

qa = load_qa_dataset()
qa_sample = random.Random(CONFIG.seed).sample(qa, min(CONFIG.evaluation.sample_size, len(qa)))

# BiomedCLIP index must also exist here (run scripts/build_index.py --retriever biomedclip).
rows = []
for name in ['biomedclip', 'colpali']:
    try:
        r = build_retriever(name); r.load_index()
        rows.append(evaluate_retriever(r, qa_sample))
    except Exception as exc:
        print(f'{name} skipped: {exc}')
pd.DataFrame(rows)

## 6 - Download the ColPali index back to your machine

In [ ]:
import shutil
shutil.make_archive('colpali_index', 'zip', 'data/index/colpali')
try:
    from google.colab import files
    files.download('colpali_index.zip')   # unzip into data/index/colpali/ locally
except ImportError:
    print('Not on Colab - archive saved as colpali_index.zip')